# NeuroMPS — the demo, as a notebook

A provenance-tracked dataset of public MEA recordings, and a calibrated
two-compartment spiking-network twin. This is the Streamlit demo's content in a form
Kaggle can host: the app is a server, and a notebook has no port for a browser to
reach, so the interactive parts here are Vega-Lite charts that carry their own data
and keep working in saved output with no kernel behind them.

> **Research decision-support only.** Nothing here is an efficacy, safety, diagnostic
> or clinical claim, and nothing here should be used to make one.

Every figure is computed from the attached files at run time. Nothing is typed in.
The full analysis, technical report and the Streamlit app are in the repository.


## Setup


In [ ]:
# Find the data whether this runs on Kaggle or in a checkout of the repository.
import json
from pathlib import Path

import numpy as np
import pandas as pd


def _find(name, roots=("/kaggle/input", ".", "..", "../..")):
    """First match for `name` under any root. Kaggle mounts datasets under
    /kaggle/input/<dataset-name>/; a checkout has them under outputs/demo and
    data/processed."""
    for root in roots:
        base = Path(root)
        if not base.exists():
            continue
        for candidate in sorted(base.rglob(name)):
            return candidate
    raise FileNotFoundError(
        f"{name} not found. On Kaggle, attach the NeuroMPS demo dataset; "
        "locally, run `python run_all.py` first (outputs/demo is written by the demo stage)."
    )


META = json.loads(_find("meta.json").read_text())
GRID = pd.read_parquet(_find("grid.parquet"))
RECORDS = pd.read_parquet(_find("real_records.parquet"))
POSTERIORS = np.load(_find("real_posteriors.npz"))["samples"]
METRICS = json.loads(_find("metrics.json").read_text())
ASSET = pd.read_parquet(_find("neuromps.parquet"))

print(f"asset            {len(ASSET):,} records")
print(
    f"simulated grid   {len(GRID)} points, {META['simulated_duration_s']:g}s each, "
    f"{META['n_per_compartment']} neurons per compartment"
)
print(f"real recordings  {len(RECORDS)} with posteriors of shape {POSTERIORS.shape}")

## 1. The data asset

Eight public sources under licences permitting reuse, every record carrying its
source, licence and a checksum of the file it came from. The split that matters is
not the count but the **data level**: only records holding spike times can be
recomputed through one common feature definition, and almost all of the modelling
uses those alone.


In [ ]:
import altair as alt

# Vega-Lite specifications carry their own data, so these stay interactive in the
# saved notebook with no kernel behind them.
by_source = ASSET.groupby(["source", "data_level"]).size().reset_index(name="records")
chart = (
    alt.Chart(by_source)
    .mark_bar()
    .encode(
        x=alt.X("records:Q", scale=alt.Scale(type="symlog"), title="records (log scale)"),
        y=alt.Y("source:N", sort="-x", title=None),
        color=alt.Color("data_level:N", title="data level"),
        tooltip=["source", "data_level", "records"],
    )
    .properties(height=220, title="REAL records by source and data level")
)
chart

In [ ]:
levels = ASSET["data_level"].value_counts()
spikes = int(levels.get("spike_times", 0))
print(f"{spikes:,} records hold spike times and can be recomputed from scratch.")
print(f"{len(ASSET) - spikes:,} hold someone else's fitted summary and cannot.")
print()
# The schema nests cell provenance, so species is a dotted field rather than a
# bare column - the kind of thing that only shows up when the notebook is run.
print("Species:", dict(ASSET["cell_source.species"].value_counts()))

## 2. Can you read a drug's mechanism from the features?

Every model works on the **change** between a treated well and its own baseline, and
splits group by culture so a model cannot recognise the culture instead of the
treatment. Within one platform there is real signal. On a platform the model has
never seen, there is close to none — and that second number is the more informative
of the two.


In [ ]:
task1 = METRICS["mechanism_baselines"]["task1_within_platform"]
task1b = METRICS["mechanism_baselines"]["task1b_independent_platform"]

rows = []
for label, block in (("within platform", task1), ("unseen platform", task1b)):
    for name, model in block["models"].items():
        low, high = model["balanced_accuracy_ci95"]
        rows.append(
            {
                "setting": label,
                "model": name.replace("_", " "),
                "balanced accuracy": model["balanced_accuracy_mean"],
                "low": low,
                "high": high,
                "chance": block["chance_balanced_accuracy"],
            }
        )
scores = pd.DataFrame(rows)

base = alt.Chart(scores).encode(y=alt.Y("model:N", title=None))
chart = (
    (
        base.mark_point(filled=True, size=90).encode(
            x=alt.X("balanced accuracy:Q", scale=alt.Scale(domain=[0, 1])),
            tooltip=list(scores.columns),
        )
        + base.mark_rule().encode(x="low:Q", x2="high:Q")
        + base.mark_tick(color="firebrick", thickness=2).encode(x="chance:Q")
    )
    .facet(row=alt.Row("setting:N", title=None))
    .properties(title="Mechanism classification, REAL data (red tick = chance)")
)
chart

## 3. The digital twin

Two compartments of leaky integrate-and-fire neurons with spike-frequency adaptation,
wired through a directional pathway, measured by virtual electrodes through the same
feature code the real recordings use. The grid below was simulated in advance; use
the dropdown to change which feature is plotted.


In [ ]:
# The app's three sliders snap to this grid. Here the same choice is a dropdown
# inside the chart, which works client-side in the saved notebook.
features = [c for c in GRID.columns if c.startswith("c0_")]
long = GRID.melt(
    id_vars=["forward_backward_ratio", "gabaa_scale", "ampa_scale", "silent"],
    value_vars=features,
    var_name="feature",
    value_name="value",
)
long["feature"] = long["feature"].str.removeprefix("c0_")

pick = alt.binding_select(options=sorted(long["feature"].unique()), name="feature ")
selection = alt.selection_point(fields=["feature"], bind=pick, value="mfr_hz")

chart = (
    alt.Chart(long)
    .mark_line(point=True)
    .encode(
        x=alt.X("gabaa_scale:Q", title="gabaa_scale  (1.0 = untreated)"),
        y=alt.Y("value:Q", title="SIMULATED feature value"),
        color=alt.Color("ampa_scale:N", title="ampa_scale"),
        tooltip=["feature", "gabaa_scale", "ampa_scale", "forward_backward_ratio", "value"],
    )
    .add_params(selection)
    .transform_filter(selection)
    .properties(width=260, height=200, title="SIMULATED twin: one grid point per marker")
    .facet(column=alt.Column("forward_backward_ratio:N", title="forward/backward ratio"))
)
chart

### One grid point's raster

Static, because a raster is a picture rather than a selection. Change `POINT` and
re-run the cell if you are running this notebook rather than reading it.


In [ ]:
import matplotlib.pyplot as plt

# One grid point's spike raster. Change `POINT` and re-run if you are running
# this notebook rather than reading it. Spikes are stored per compartment.
POINT = 2
row = GRID.iloc[POINT]
run = np.load(_find(row["run_file"]))
offset = int(META["n_per_compartment"])
fig, ax = plt.subplots(figsize=(9, 3.2))
for c, colour in ((0, "#0072B2"), (1, "#D55E00")):
    ax.plot(
        run[f"c{c}_times"],
        run[f"c{c}_neurons"] + (0 if c == 0 else offset),
        "|",
        markersize=2,
        color=colour,
        label=f"compartment {c}",
    )
ax.legend(loc="upper right", fontsize=8, markerscale=4)
ax.set_xlabel("time (s)")
ax.set_ylabel("neuron")
ax.set_title(
    f"SIMULATED raster - gabaa_scale {row['gabaa_scale']:g}, "
    f"ampa_scale {row['ampa_scale']:g}, ratio {row['forward_backward_ratio']:g}",
    fontsize=10,
)
plt.show()

## 4. Running the twin backwards on real recordings

A posterior trained only on simulations, applied to real bicuculline recordings. Two
things to look for, and both are negative results: untreated recordings do not come
back at the neutral value of 1.0, and the inferred scale does not track dose. The
report's §18 shows the second is a property of the measurement rather than of the
estimator — no feature in this set moves monotonically with dose.


In [ ]:
# The posterior's inferred GABA-A scale for each real recording, against dose.
names = META["parameter_names"]
# NaN where rejection sampling timed out before drawing the full set, so the
# median must ignore them rather than propagate.
gabaa = np.nanmedian(POSTERIORS[:, :, names.index("gabaa_scale")], axis=1)
doses = RECORDS["dose_value"].fillna(0.0).to_numpy(dtype=float)
doses = np.where(RECORDS["dose_unit"].fillna("uM").to_numpy() == "nM", doses / 1000.0, doses)

inferred = pd.DataFrame(
    {
        "dose_uM": doses,
        "gabaa_scale": gabaa,
        "device": RECORDS["device_id"],
        "phase": RECORDS["phase"],
    }
)
chart = (
    alt.Chart(inferred)
    .mark_circle(size=70, opacity=0.7)
    .encode(
        x=alt.X("dose_uM:Q", title="bicuculline dose (uM)"),
        y=alt.Y("gabaa_scale:Q", title="inferred gabaa_scale  (1.0 = untreated)"),
        color=alt.Color("phase:N"),
        tooltip=["device", "dose_uM", "gabaa_scale", "phase"],
    )
    .properties(
        width=520,
        height=260,
        title="Posterior trained on SIMULATIONS, applied to REAL recordings",
    )
)
chart + alt.Chart(pd.DataFrame({"y": [1.0]})).mark_rule(strokeDash=[4, 4]).encode(y="y:Q")

## What this does not show

- Nothing clinical, and no efficacy or safety claim.
- Mechanism accuracy is modest within a platform and near chance on an unseen one.
- Most classification rests on 73 treated/baseline pairs from 50 cultures.
- The twin matches the real band on some features and not others.

The simulated grid here was run at {META['n_per_compartment']} neurons per
compartment — printed by the setup cell, and worth comparing against the calibrated
density reported in the technical report, since cell density is what sets the twin's
synchrony regime.
